In [19]:
from typing import TypedDict, Annotated

from dotenv import load_dotenv
from langchain_groq import ChatGroq
import langchain,langchain_groq,langgraph
from langgraph.graph import StateGraph, START

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)


In [20]:
from langgraph.types import interrupt
from langgraph.graph import add_messages
from langchain_core.messages import BaseMessage, AIMessage


class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


def takeApprovalAndContinue(chatState: ChatState):
    question = chatState["messages"][-1].content
    decision = interrupt(f"Approve this conversation with the following question?\n{question}")
    print(decision)
    if decision == "yes":
        llm_ans = llm.invoke(question)
        return {"messages": [llm_ans]}
    else:
        return {"messages": [AIMessage(content="Not approved!")]}




In [21]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.constants import END

graph = StateGraph(ChatState)

graph.add_node("takeApprovalAndContinue",takeApprovalAndContinue)
graph.add_edge(START,"takeApprovalAndContinue")
graph.add_edge("takeApprovalAndContinue",END)
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

In [22]:
from langchain_core.messages import HumanMessage
from pprint import pprint
config = {
        "configurable": {
            "thread_id": "1"
        }
    }
result = app.invoke({
    "messages": [
        HumanMessage(content="Who is Alakh Pandey?")
    ]
},    config=config)


pprint(result, width=120)

{'__interrupt__': [Interrupt(value='Approve this conversation with the following question?\nWho is Alakh Pandey?',
                             id='d436ebcda7291f53e1b8f1c051619230')],
 'messages': [HumanMessage(content='Who is Alakh Pandey?', additional_kwargs={}, response_metadata={}, id='8ddc9aba-fddf-4dbb-a91a-65e5035bd161')]}


In [23]:
interrupt_message = result["__interrupt__"][0].value
user_input = input(interrupt_message)

In [24]:
from langgraph.types import Command

final_result = app.invoke(Command(resume=user_input),config=config)
final_result["messages"][-1].content

yes


'**Alakh Pandey** (often called **“Physics Wallah”**) is a well‑known Indian educator, YouTuber, and entrepreneur who has become one of the most popular figures in the country’s online‑learning space.\n\n| Aspect | Details |\n|--------|---------|\n| **Full name** | Alakh Pandey |\n| **Born** | 1990 (approx.) in Prayagraj (formerly Allahabad), Uttar\u202fPradesh, India |\n| **Education** | • B.Sc. (Physics) – University of Allahabad<br>• M.Sc. (Physics) – University of Allahabad (did not complete the degree, left to focus on teaching) |\n| **Career start** | Began tutoring classmates in high school; later started a small coaching centre in his hometown. In 2016 he launched a YouTube channel called **“Physics Wallah”** where he posted free video lessons in Hindi for Class\u202f11‑12 physics, chemistry, and mathematics. |\n| **YouTube channel** | • Launched: 2016<br>• Content: Full‑syllabus lectures for Indian CBSE/ICSE and state board students, plus exam‑strategy videos.<br>• Subscribers